In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()


customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     
    (104, 99, "2024-01-04", 1500),    
    (105, 1, "2024-01-05", None),    
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])


orders = orders.withColumn("order_date", to_date(col("order_date")))

In [0]:
# TODO 1: Clean data
# - Remove nulls
# - Handle negative values
# - Trim names

# TODO 2: Validate data
# - Find invalid customer_id using left_anti join

# TODO 3: Join datasets

# TODO 4: Apply transformations
# - total spend per customer
# - count orders

# TODO 5: Window functions
# - rank customers by spend

# TODO 6: Save output
# final_df.write.mode("overwrite").csv("/tmp/phase6_output")

In [0]:
customers.show()
orders.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          3|    NULL|  bob@example.com|Bangalore|
|          4|   David|             NULL|   Mumbai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     105|          1|2024-01-05|  NULL|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
customers=customers.replace("NULL",None)
customers.show()
customers=customers.fillna({
    "name":"Unknown",
    "email":"Unknown"
})

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          3|    NULL|  bob@example.com|Bangalore|
|          4|   David|             NULL|   Mumbai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+



In [0]:
customers.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          3| Unknown|  bob@example.com|Bangalore|
|          4|   David|          Unknown|   Mumbai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+



In [0]:
joined_df=customers.join(orders,on="customer_id",how="right")
joined_df.show()
joined_df=joined_df.withColumn("Data Validation",
                     when(col("name").isNull(),"Invalid")\
                    .when(col("amount")<=0,"Invalid")\
                        .when(col("amount").isNull(),"Invalid")\
                            .otherwise("Valid")
                        
                             )
joined_df.show()

+-----------+--------+-----------------+---------+--------+----------+------+
|customer_id|    name|            email|     city|order_id|order_date|amount|
+-----------+--------+-----------------+---------+--------+----------+------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|
|          2|  Alice |alice@example.com|  Chennai|     102|2024-01-02|  2000|
|          3| Unknown|  bob@example.com|Bangalore|     103|2024-01-03|  -500|
|         99|    NULL|             NULL|     NULL|     104|2024-01-04|  1500|
|          1|John Doe| john@example.com|Hyderabad|     105|2024-01-05|  NULL|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|
+-----------+--------+-----------------+---------+--------+----------+------+

+-----------+--------+-----------------+---------+--------+----------+------+---------------+
|customer_id|    name|            email|     ci

In [0]:
joined_df=joined_df.withColumn("name",trim(col("name")))
joined_df.show()

+-----------+--------+-----------------+---------+--------+----------+------+---------------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Data Validation|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|          Valid|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|          Valid|
|          3| Unknown|  bob@example.com|Bangalore|     103|2024-01-03|  -500|        Invalid|
|         99|    NULL|             NULL|     NULL|     104|2024-01-04|  1500|        Invalid|
|          1|John Doe| john@example.com|Hyderabad|     105|2024-01-05|  NULL|        Invalid|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|          Valid|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|          Valid|
+-----------+--------+-----------------+---------+--------+-

**WINDOW FUNCTIONS**


**Dropped invalid rows**

In [0]:
joined_df=joined_df.filter(col("Data Validation")!="Invalid")
joined_df.show()

+-----------+--------+-----------------+---------+--------+----------+------+---------------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Data Validation|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|          Valid|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|          Valid|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|          Valid|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|          Valid|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+



In [0]:
from pyspark.sql.window import Window
window=Window.partitionBy("name").orderBy("order_id")
joined_df=joined_df.withColumn("total_spend",sum(col("amount")).over(window))
joined_df.orderBy("customer_id").show()

+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Data Validation|total_spend|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|          Valid|       1000|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|          Valid|       2000|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|          Valid|       3000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|          Valid|       6000|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+



top 3 customers by city


In [0]:
window=Window.partitionBy("city").orderBy("total_spend")
joined_df=joined_df.select("name","city",rank().over(window).alias("rank")).alias("top 3 customers")
joined_df.show()

+--------+---------+----+
|    name|     city|rank|
+--------+---------+----+
|    NULL|     NULL|   1|
| Unknown|Bangalore|   1|
|  Alice |  Chennai|   1|
|John Doe|Hyderabad|   1|
|John Doe|Hyderabad|   1|
|     Eva|Hyderabad|   3|
|     Eva|Hyderabad|   4|
+--------+---------+----+



In [0]:
window=Window.orderBy(col("total_spend").desc())
final_df=joined_df.select("name","city","total_spend",rank().over(window).alias("rank"))
final_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+---------+-----------+----+
|    name|     city|total_spend|rank|
+--------+---------+-----------+----+
|     Eva|Hyderabad|       6000|   1|
|     Eva|Hyderabad|       3000|   2|
|   Alice|  Chennai|       2000|   3|
|John Doe|Hyderabad|       1000|   4|
+--------+---------+-----------+----+



In [0]:
joined_df.show()

+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Data Validation|total_spend|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|          Valid|       2000|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|          Valid|       3000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|          Valid|       6000|
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|          Valid|       1000|
+-----------+--------+-----------------+---------+--------+----------+------+---------------+-----------+



In [0]:
window=Window.partitionBy("name").orderBy("order_date")
previous_order_tab=joined_df.select("name","city","order_date",lag("order_date",1).over(window).alias("previous_order"))
previous_order_tab.show()

+--------+---------+----------+--------------+
|    name|     city|order_date|previous_order|
+--------+---------+----------+--------------+
|   Alice|  Chennai|2024-01-02|          NULL|
|     Eva|Hyderabad|2024-01-06|          NULL|
|     Eva|Hyderabad|2024-01-07|    2024-01-06|
|John Doe|Hyderabad|2024-01-01|          NULL|
+--------+---------+----------+--------------+



**DATE ANALYSIS**

In [0]:
joined_df.select("name","order_date",month(col("order_date")).alias("Month of the Order")).show()

+--------+----------+------------------+
|    name|order_date|Month of the Order|
+--------+----------+------------------+
|John Doe|2024-01-01|                 1|
|   Alice|2024-01-02|                 1|
|     Eva|2024-01-06|                 1|
|     Eva|2024-01-07|                 1|
+--------+----------+------------------+



In [0]:
joined_df.groupBy(month(col("order_date"))).agg(
    sum(col("amount")).alias("Monthly Sales Aggregation")
).show()

+-----------------+-------------------------+
|month(order_date)|Monthly Sales Aggregation|
+-----------------+-------------------------+
|                1|                     9000|
+-----------------+-------------------------+



In [0]:
previous_order_tab.select("name","order_date","previous_order",when(col("previous_order").isNotNull(),datediff(col("order_date"),col("previous_order"))).alias("Difference between Days")).show()

+--------+----------+--------------+-----------------------+
|    name|order_date|previous_order|Difference between Days|
+--------+----------+--------------+-----------------------+
|   Alice|2024-01-02|          NULL|                   NULL|
|     Eva|2024-01-06|          NULL|                   NULL|
|     Eva|2024-01-07|    2024-01-06|                      1|
|John Doe|2024-01-01|          NULL|                   NULL|
+--------+----------+--------------+-----------------------+



In [0]:
joined_df.groupBy(dayofmonth(col("order_date"))).agg(
    sum(col("amount")).alias("Monthly Sales Aggregation")
).show()

+----------------------+-------------------------+
|dayofmonth(order_date)|Monthly Sales Aggregation|
+----------------------+-------------------------+
|                     1|                     1000|
|                     2|                     2000|
|                     6|                     3000|
|                     7|                     3000|
+----------------------+-------------------------+

